In [ ]:
import h5py as h5
from pathlib import Path
from histpy import Histogram, Axes, Axis
import scoords
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from cosipy.response.relative_coordinates import RelativeCDSCoordinates


wdir = Path("/Users/imartin5/cosi/scratch/response_relative_coordinates/v4")

## Total effective area

In [ ]:
file = h5.File(wdir/'ResponseContinuum.rel.binnedimaging.imagingresponse.nonsparse.aeff.h5')
drm = file['DRM']

In [ ]:
counts = np.asarray(drm['COUNTS'], dtype = float)

In [ ]:
axes = Axes.open(drm["AXES"])

In [ ]:
h = Histogram(axes, contents = counts, dtype = float, copy_contents = True)

In [ ]:
# To Aeff 

aeff_corr = drm['EFF_AREA'] # Weight to convert to Aeff. Already divided by NuLambda npix

h *= axes.expand_dims(aeff_corr, axes.label_to_index("Ei"))

# Aeff above is in cm2
h.to(u.cm*u.cm, update = False, copy = False)

In [ ]:
hnorm = h.project('NuLambda', 'Ei')

In [ ]:
hnorm.write(wdir/'ResponseContinuum.area.relative.nonsparse.h5', "AEFF", overwrite = True)

In [ ]:
hnorm.slice[{'NuLambda':0}].project('Ei').plot()

In [ ]:
hnorm.slice[{'Ei':hnorm.axes['Ei'].find_bin(5*u.MeV)}].project('NuLambda').plot()

## Differential area

In [ ]:
file = h5.File(wdir/'ResponseContinuum.rel.binnedimaging.imagingresponse.nonsparse.h5')

In [ ]:
drm = file['DRM']

In [ ]:
file['DRM'].keys()

In [ ]:
counts = np.asarray(drm['COUNTS'], dtype = float)

In [ ]:
ntot = np.sum(counts)
ntot

In [ ]:
axes = Axes.open(drm["AXES"])

In [ ]:
axes.labels

In [ ]:
phi_edges_mesh, arm_edges_mesh, az_edges_mesh = np.meshgrid(axes['Phi'].edges.to_value(u.rad), 
                                                            axes['Theta'].edges.to_value(u.rad), 
                                                            axes['Zeta'].edges.to_value(u.rad), indexing = 'ij')

phase_space_cds = RelativeCDSCoordinates.get_relative_cds_phase_space(phi_edges_mesh[:-1, :-1, :-1],
                                  phi_edges_mesh[ 1:, :-1, :-1],
                                  arm_edges_mesh[:-1, :-1, :-1],
                                  arm_edges_mesh[:-1, 1:,  :-1:],
                                   az_edges_mesh[:-1, :-1, :-1],
                                   az_edges_mesh[:-1, :-1,  1:])

ei_centers_mesh, em_widths_mesh = np.meshgrid(axes['Ei'].centers.to_value(u.keV), 
                                             axes['Epsilon'].widths.to_value(''), 
                                             indexing = 'ij')

phase_space_em = ei_centers_mesh * em_widths_mesh

In [ ]:
npix_phys = np.sum(axes.broadcast(phase_space_cds, ['Phi', 'Theta', 'Zeta']) > 0)

In [ ]:
h = Histogram(axes, contents = counts, dtype = float, copy_contents = True)

In [ ]:
def mean_dist(vs_axes, dist_axes, expand = True):
    
    all_axes = np.concatenate([vs_axes, dist_axes])
    
    dist = h.project(all_axes)
    
    if len(vs_axes) > 0:
        norm = dist.axes.expand_dims(h.project(vs_axes), vs_axes)
    else:
        norm = np.sum(h)
        
    dist /= norm

    if expand:
        dist = h.axes.expand_dims(dist, all_axes)
    
    return dist
    
epsilon_dist = mean_dist(['Ei'], ['Epsilon'])
phi_theta_dist = mean_dist(['Ei'], ['Phi', 'Theta'])
zeta_dist = mean_dist(['Ei'], ['Zeta'])

cds_dist = phi_theta_dist * zeta_dist

trig_counts = h.project(['NuLambda','Ei'])

mean_response = (axes.expand_dims(trig_counts, ['NuLambda','Ei']) * epsilon_dist * cds_dist)

# # Poisson with Gamma prior (mu = mean, stdev = sqrt(k * mu))
# k = 1 # Weight mean vs actual counts
# mean_response = (h.contents + mean_response * k)/(1 + k)

# Poisson with Gamma prior (mu = mean, stdev = sqrt(k) * n)
k = 0 # Weight mean vs actual counts
mean_response = (h.contents**3 + k*mean_response**2)/(h.contents**2 + k*mean_response)

# # Keep same normalization
h[:] = mean_response * axes.expand_dims(trig_counts, ['NuLambda','Ei']) / axes.expand_dims(np.nansum(mean_response, axis = (2,3,4,5)), ['NuLambda','Ei'])

In [ ]:
# Should be 0 --i.e. no event with 0 phase space and >0 events
np.sum(np.isinf(h.contents))

In [ ]:
# It's ok to have bins with 0 phase space or even 0 for the mean response
print(counts.size)
print(counts.size - npix_phys)
print(np.sum(np.isnan(h.contents)))

In [ ]:
import sys

# Fix unphysical bins (0 phasespace) or mean=0
h[np.isnan(h.contents)] = 0

In [ ]:
# To Aeff 

aeff_corr = drm['EFF_AREA'] # Weight to convert to Aeff. Already divided by NuLambda npix

h *= axes.expand_dims(aeff_corr, axes.label_to_index("Ei"))

# Aeff above is in cm2
h.to(u.cm*u.cm, update = False, copy = False)

In [ ]:
hnorm = h.project('NuLambda','Ei')

In [ ]:
h.axes.labels

In [ ]:
h.axes.units

In [ ]:
# Phase space
h /= axes.expand_dims(phase_space_cds, axes.label_to_index(['Phi', 'Theta', 'Zeta']))
h /= axes.expand_dims(phase_space_em, axes.label_to_index(['Ei', 'Epsilon']))

In [ ]:
# Should be 0 --i.e. no event with 0 phase space and >0 events
np.sum(np.isinf(h.contents))

In [ ]:
# Shoulds be the same at this point
print(counts.size - npix_phys)
print(np.sum(np.isnan(h.contents)))

In [ ]:
import sys

# Fix unphysical bins (0 phasespace)
h[np.isnan(h.contents)] = 0 * h.unit

In [ ]:
ax,_ = (h.slice[{'Ei':5,'NuLambda':0, 'Phi':5, 'Epsilon':11, 'Zeta':9}].project('Theta')).plot()
ax,_ = h.slice[{'Ei':5}].project('NuLambda', 'Zeta').plot()
ax.set_xlim(0,100)
h.slice[{'Ei':6,'NuLambda':h.axes['NuLambda'].ang2pix(90, 30, lonlat = True)}].project('Zeta','Phi').plot()

In [ ]:
ax,_ = (h.project('Phi','Theta')).plot()

In [ ]:
ax,_ = (h.slice[{'Ei':5,'NuLambda':480, 'Phi':8, 'Epsilon':11}].project('Zeta','Theta')).plot()

In [ ]:
h.axes['Ei'].edges

In [ ]:
h.axes['Phi'].edges

In [ ]:
h.slice[{'Ei':7,'NuLambda':0, 'Phi':8}].project('Theta').plot()

In [ ]:
h.slice[{'Ei':7,'NuLambda':0, 'Phi':8}].project('Zeta').plot()

In [ ]:
hproj = h.slice[{'Ei':5,'NuLambda':480, 'Phi':8, 'Epsilon':11, 'Zeta':13}].project('Theta')

ax,_ = hproj.plot()

# ax.set_yscale('log')
# ax.set_ylim(np.max(hproj)*1e-2, np.max(hproj)*1.1)

ax.set_xlim(-5,5)

In [ ]:
np.prod(h.shape)/1e5/36000

In [ ]:
hnorm.slice[{'NuLambda':0}].project('Ei').plot()

In [ ]:
hnorm.axes['Ei'].edges

In [ ]:
hnorm.slice[{'Ei':5}].project('NuLambda').plot()

In [ ]:
slice(0,100)

In [ ]:
dist_top = h.slice[{'NuLambda':slice(None,100)}].project('Theta')
dist_bottom = h.slice[{'NuLambda':slice(300,400)}].project('Theta')

dist_top /= np.sum(dist_top)
dist_bottom /= np.sum(dist_bottom)


ax,_ = dist_top.plot()
dist_bottom.plot(ax)

ax.set_xlim(-20,20)

In [ ]:
eps_top = h.slice[{'NuLambda':slice(None,100)}].project('Epsilon')
eps_bottom = h.slice[{'NuLambda':slice(668,768)}].project('Epsilon')

eps_top /= np.sum(eps_top)
eps_bottom /= np.sum(eps_bottom)


ax,_ = eps_top.plot()
eps_bottom.plot(ax)

ax.set_xlim(-.1,.1)

In [ ]:
h.project('Epsilon').plot()

In [ ]:
ax,_ = h.project('Theta').plot()

#ax.set_xlim(-10,10)

In [ ]:
h.project('Phi').plot()

In [ ]:
h.project('Zeta').plot()

In [ ]:
h.axes['NuLambda'].pix2skycoord(180)

In [ ]:
from matplotlib.colors import LogNorm

hspec = h.slice[{'NuLambda':0}].project('Ei','Epsilon')

hspec.plot(norm=LogNorm(vmin=np.max(hspec).value*1e-3, vmax=np.max(hspec).value))

In [ ]:
ax,_ = h.slice[{'Ei':7,'NuLambda':180}].project('Epsilon').plot()

ax.set_xlim(-.1,.1)

In [ ]:
h.slice[{'Ei':6,'NuLambda':180}].project('Zeta').rebin(2).plot()

In [ ]:
ax,_ = h.slice[{'Ei':5}].project('NuLambda', 'Zeta').plot()
ax.set_xlim(0,100)

In [ ]:
h.slice[{'Ei':6,'NuLambda':h.axes['NuLambda'].ang2pix(20, 30, lonlat = True)}].project('Zeta','Phi').plot()

In [ ]:
(h.slice[{'Ei':5,'NuLambda':180}].project('Phi')).plot()

In [ ]:
hphiei = (h.slice[{'NuLambda':0}].project('Ei','Phi'))

hphiei.plot()

hphiei.plot(norm=LogNorm(vmin=np.max(hphiei).value*1e-3, vmax=np.max(hphiei).value))

In [ ]:
# Corase approximation to get a good binnning scheme

from scipy.stats import norm

ax,_ = (h.slice[{'Ei':5,'NuLambda':0}].project('Theta')).plot()

pdf1 = 1200*norm(scale = 1).pdf(np.linspace(-30,30,100))
pdf2 = 4e3*norm(scale = 5).pdf(np.linspace(-30,30,100))
pdf3 = 2e3*norm(scale = 25).pdf(np.linspace(-30,30,100))

ax.plot(np.linspace(-30,30,100), pdf1)

ax.plot(np.linspace(-30,30,100), pdf2)

ax.plot(np.linspace(-30,30,100), pdf3)


x = np.linspace(-180,180,1000)
pdf_arm = .5*1200*norm(scale = 1).pdf(x) + +.5*4e3*norm(scale = 5).pdf(x) + 2e3*norm(scale = 25).pdf(x)


ax.plot(x, 5*pdf_arm)

ax.set_xlim(-50,30)

In [ ]:
x = np.linspace(-180,180,1000)
pdf_arm = .5*1200*norm(scale = 1).pdf(x) + +.5*4e3*norm(scale = 5).pdf(x) + 2e3*norm(scale = 25).pdf(x)
x[np.searchsorted(np.cumsum(pdf_arm/np.sum(pdf_arm)), np.concatenate([[.001,.01],np.linspace(.1,.9,8),[.99,.999]]))]

In [ ]:
h.axes['Phi'].edges

In [ ]:
axes.labels

In [ ]:
axes['Epsilon'].edges

In [ ]:
axes.labels

In [ ]:
np.nansum(counts[488, 0])

In [ ]:
hproj = h.slice[{'Ei':0,'NuLambda':488}].project_out('NuLambda', 'Ei')

In [ ]:
(h.slice[{'Ei':6,'NuLambda':488, 'Phi':6, 'Theta':5}].project('Epsilon','Zeta')).plot()

In [ ]:
h.axes['Epsilon'].edges

In [ ]:
(h.slice[{'Ei':5,'NuLambda':1, 'Phi':16, 'Epsilon':11, 'Zeta':16}].project('Theta')).plot()

In [ ]:
(h.slice[{'Ei':5,'NuLambda':180, 'Phi':16}].project('Theta')).plot()

In [ ]:
(h.slice[{'Ei':5,'NuLambda':180}].project('Phi','Theta')).plot()

In [ ]:
rng = np.random.default_rng()
nsamples = int(1e6)
samples_nulambda = SkyCoord(lon = rng.uniform(0, 1, size = nsamples), lat = rng.uniform(0, 1, size = nsamples), unit='rad', frame = 'spacecraftframe')
samples_ei = u.Quantity(rng.uniform(200, 2000, size = nsamples), u.keV)
samples_eps = u.Quantity(rng.uniform(.5,1.2,size = nsamples)) # Eps currently ios Unit(dimensionless). Needs fix.
samples_phi = u.Quantity(rng.uniform(.5,1.2,size = nsamples), u.rad)
samples_theta = u.Quantity(rng.uniform(0,1,size = nsamples), u.rad)
samples_zeta = u.Quantity(rng.uniform(0,1,size = nsamples), u.rad)

In [ ]:
%%time
h.interp(samples_nulambda, samples_ei, samples_eps, samples_phi, samples_theta, samples_zeta)

In [ ]:
np.max(h.interp(samples_nulambda, samples_ei, samples_eps, samples_phi, samples_theta, samples_zeta))

In [ ]:
%%time
hnorm.interp(samples_nulambda, samples_ei)

In [ ]:
from cosipy.polarization import PolarizationAxis
new_axis = PolarizationAxis(h.axes['Zeta'], convention = 'RelativeX')
new_axis.copy()._convention

In [ ]:
# Bring back phase space and store to disk
h *= axes.expand_dims(phase_space_cds, axes.label_to_index(['Phi', 'Theta', 'Zeta']))
h *= axes.expand_dims(phase_space_em, axes.label_to_index(['Ei', 'Epsilon']))

# Specify convention
from cosipy.polarization import PolarizationAxis
h.axes['Zeta'] = PolarizationAxis(h.axes['Zeta'], convention = 'RelativeX')

In [ ]:
h.axes['Zeta']._convention

In [ ]:
h.write(wdir/'ResponseContinuum.area.relative.nonsparse.h5', "IRF", overwrite = True)

In [ ]:
h.axes.labels

In [ ]:
h.slice[{'NuLambda':0}].project('Ei').plot()

In [ ]:
h.axes['Ei'].edges

In [ ]:
from matplotlib import pyplot as plt
plt.plot(h.slice[{'NuLambda':0, 'Ei':5}].project('Zeta').contents)
plt.ylim(0, None)

In [ ]:
h.axes['Ei'].axis_scale